# Compare Performance

In [ ]:
!ls ../../../results

In [ ]:
from hglm.compare_analyses import load_update_all, extract, plot_feats

folder = '/home/matt/Dropbox/pnl_hglm/results/exp_24Aug09-1616/'
df = load_update_all(folder)
pval_list, seed_list, score_dict = extract(df)

In [ ]:
fig = plot_feats(pval_list, score_dict, feat_list=('auc', 'f1', 'sens', 'spec'))

fig.set_size_inches(15, 6)
fig.tight_layout()

# Examine case where TFCE has greatest advantage of HGLM



In [ ]:
import numpy as np
from hglm.compare_analyses import load

feat = 'f1'

feat_diff_dict = dict()
for feat in ('f1', 'sens', 'spec', 'auc'):
    feat_diff_dict[feat] = score_dict['AnalysisHGLM', feat] - score_dict['AnalysisTFCE', feat]

# pval_seed_sorted is a list of (pval, seed) tuples, sorted in increasing "feat" (see above)
feat_diff = feat_diff_dict[feat]
seed_idx, pval_idx = np.unravel_index(np.argsort(feat_diff.flatten()), feat_diff.shape)
n_not_nan = (~np.isnan(feat_diff)).sum()
pval_seed_sorted = [(pval_list[pval_idx[idx]], seed_list[seed_idx[idx]]) for idx in range(n_not_nan)]

In [ ]:
np.nanmin(feat_diff_dict['f1'])

In [ ]:
# get the worst case scenario
p_val, seed = pval_seed_sorted[0]
ana_hglm, effect = load(folder=folder, df=df, p_val=p_val, seed=seed, Analysis='AnalysisHGLM')

In [ ]:
from hglm.plot import scatter_plotly

fig, df_reg = scatter_plotly(ana_hglm, mask_target=effect.mask, plot_permute=False)

Each row below represents a region which is "discovered": they were chosen as representatives of all the significant regions.

In [ ]:
df_reg[df_reg['discovered']]

In [ ]:
fig.show()

# Compare Computation Time

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.swarmplot(data=df[df['Analysis'] != 'AnalysisHGLM-maxF1'], x='time_sec', hue='Analysis', size=3)
plt.suptitle('Time per experiment')
plt.xlabel('Time (seconds)')
plt.gcf().set_size_inches(10, 5)

# Roughness vs Performance Difference


In [ ]:
feat = 'f1'
rough_list = list()
pval_list = list()
diff_list = list()
for (seed, pval), _df in df.groupby((['seed', 'p_val'])):
    if _df.shape[0] != 2:
        continue
    row_hglm = _df[_df['Analysis']=='AnalysisHGLM'].iloc[0, :]
    row_tfce = _df[_df['Analysis']=='AnalysisTFCE'].iloc[0, :]

    assert np.isclose(row_hglm['rough'], row_tfce['rough'])
    rough_list.append(row_hglm['rough'])
    pval_list.append(np.log10(pval))
    diff_list.append(row_hglm[feat] - row_tfce[feat])

In [ ]:
plt.scatter(rough_list, diff_list, c=pval_list, cmap='viridis')